# Weapons & Armor

**Weapons** are tools the LLM can call during quest execution.  
**Armor** provides pre/post-processing guardrails on LLM I/O.

This notebook shows how to use the built-in ones and create your own.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

## Weapons (Tools)

Equip an adventurer with a `FileReadWeapon` so it can read local files.

In [ ]:
from guildmaster_ai import GeneralAdventurer, GuildBuilder
from guildmaster_ai.weapons.file_read import FileReadWeapon

reader = GeneralAdventurer(name="FileReader")
reader.equip_weapon(FileReadWeapon())

guild = GuildBuilder().with_llm_provider("openrouter").register_adventurer(reader).build()

# Read and summarize pyproject.toml
toml_path = os.path.join(os.getcwd(), "..", "pyproject.toml")
result = await guild.run_quest(
    f"Read '{toml_path}' and list the project's core dependencies. Be concise."
)
print(result.summary)

## Custom Weapon

Create a weapon by subclassing `BaseWeapon`. Weapons are LangChain `BaseTool` subclasses.

In [ ]:
from typing import Any

from pydantic import BaseModel, Field

from guildmaster_ai.weapons.base_weapon import BaseWeapon


class CalculatorInput(BaseModel):
    expression: str = Field(description="A Python math expression to evaluate")


class CalculatorWeapon(BaseWeapon):
    name: str = "calculator"
    description: str = "Evaluates a mathematical expression and returns the result."
    args_schema: type[BaseModel] = CalculatorInput

    async def execute(self, expression: str, **kwargs: Any) -> dict[str, Any]:
        try:
            # Only allow safe math operations
            result = eval(expression, {"__builtins__": {}}, {})
            return {"result": result}
        except Exception as e:
            return {"error": str(e)}


calc_adventurer = GeneralAdventurer(name="MathWiz")
calc_adventurer.equip_weapon(CalculatorWeapon())

guild2 = GuildBuilder().with_llm_provider("openrouter").register_adventurer(calc_adventurer).build()

result = await guild2.run_quest("What is 2**10 + 42? Use the calculator.")
print(result.summary)

## Armor (Guardrails)

Armor runs before and/or after LLM calls. The `ContentFilterArmor` blocks
messages matching forbidden patterns.

In [ ]:
from guildmaster_ai.armor.content_filter import ContentFilterArmor

guarded = GeneralAdventurer(name="Guarded")
guarded.wear_armor(ContentFilterArmor(blocked_patterns=["password", "secret"]))

guild3 = GuildBuilder().with_llm_provider("openrouter").register_adventurer(guarded).build()

# This works fine
result = await guild3.run_quest("What is 2 + 2?")
print(f"Clean quest: {result.summary}")

# This gets blocked by the armor (returns failed result, not an exception)
result = await guild3.run_quest("What is my password?")
print(f"\nBlocked: {result.summary}")
assert not result.success

## Custom Armor

Override `pre_process()` and/or `post_process()` to create custom guardrails.

In [ ]:
from guildmaster_ai.armor.base_armor import ArmorResult, BaseArmor


class UpperCaseArmor(BaseArmor):
    """Converts all LLM output to uppercase."""

    @property
    def name(self) -> str:
        return "uppercase"

    async def post_process(self, content: str) -> ArmorResult:
        return ArmorResult(verdict="pass", modified_content=content.upper())


shouter = GeneralAdventurer(name="Shouter")
shouter.wear_armor(UpperCaseArmor())

guild4 = GuildBuilder().with_llm_provider("openrouter").register_adventurer(shouter).build()

result = await guild4.run_quest("Say hello in French.")
print(result.summary)